# Lean cache health — isolated Colab replay (2026-08-01)

Fresh public clones only. No token, no push, no shared cache, and no source edits. The pre-build PR #39 oracle is an ordering probe; the decisive oracle runs follow the minimal project build.

In [ ]:
%%bash
set -uo pipefail
RUN=/content/lean-cache-health-$(date -u +%Y%m%dT%H%M%SZ)
mkdir -p "$RUN"
TRANSCRIPT="$RUN/COLAB-TRANSCRIPT.txt"
exec > >(tee -a "$TRANSCRIPT") 2>&1
export PATH="$HOME/.elan/bin:$PATH"
FAILURES=0
LAST_CODE=0

run_keep() {
  label="$1"
  shift
  start=$(date +%s)
  echo "COMMAND_START label=$label utc=$(date -u +%Y-%m-%dT%H:%M:%SZ)"
  echo "COMMAND=$*"
  set +e
  timeout 1800 "$@" 2>&1 | tee "$RUN/$label.log"
  LAST_CODE=$?
  set -e
  stop=$(date +%s)
  echo "COMMAND_END label=$label exit=$LAST_CODE elapsed_seconds=$((stop-start))"
  sha256sum "$RUN/$label.log"
  if [ "$LAST_CODE" -ne 0 ]; then FAILURES=$((FAILURES+1)); fi
}

run_probe() {
  label="$1"
  shift
  start=$(date +%s)
  echo "PROBE_START label=$label utc=$(date -u +%Y-%m-%dT%H:%M:%SZ)"
  set +e
  timeout 1800 "$@" 2>&1 | tee "$RUN/$label.log"
  LAST_CODE=$?
  set -e
  stop=$(date +%s)
  echo "PROBE_END label=$label exit=$LAST_CODE elapsed_seconds=$((stop-start))"
  sha256sum "$RUN/$label.log"
}

echo "UTC_START=$(date -u +%Y-%m-%dT%H:%M:%SZ)"
echo "RUN=$RUN"
uname -a
nproc
git --version
python3 --version

if ! command -v elan >/dev/null 2>&1; then
  curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -o "$RUN/elan-init.sh"
  sha256sum "$RUN/elan-init.sh"
  sh "$RUN/elan-init.sh" -y --default-toolchain none
fi

REPO=https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git
MAIN=b954a67538fdde62f932309af98271cf99667ec9
PR39=5172a51b0fad455d9009b1805a48b8be54acbbcc
PR43_AUDITED=0ac9b18c98cf12b305611fd74087816a3b5f1e32
git ls-remote "$REPO" refs/heads/main refs/pull/39/head refs/pull/43/head
elan --version

git clone --no-tags "$REPO" "$RUN/pr39"
cd "$RUN/pr39"
git fetch origin refs/pull/39/head
git checkout --detach "$PR39"
echo "PR39_HEAD=$(git rev-parse HEAD)"
echo "PR39_LAKE_PREEXISTS=$(test -e .lake && echo yes || echo no)"
cp lake-manifest.json "$RUN/pr39-manifest-before.json"
sha256sum lean-toolchain lakefile.lean lake-manifest.json
cat lean-toolchain
run_keep pr39-lake-update lake update
cmp -s lake-manifest.json "$RUN/pr39-manifest-before.json"
if [ "$?" -ne 0 ]; then echo "PR39_MANIFEST_UNCHANGED=no"; FAILURES=$((FAILURES+1)); else echo "PR39_MANIFEST_UNCHANGED=yes"; fi
run_keep pr39-cache-get lake exe cache get
lean --version
lake --version
printf 'import Mathlib
' > "$RUN/ImportMathlib.lean"
run_keep pr39-import-1 lake env lean "$RUN/ImportMathlib.lean"
run_keep pr39-import-2 lake env lean "$RUN/ImportMathlib.lean"
run_probe pr39-oracle-before-project-build lake env lean docs/SU2-TWO-TRANSPORTER-NOGO-ORACLE.lean
run_keep pr39-minimal-build lake build YangMills.OS.TwoTransporterHaarProjection
run_keep pr39-oracle-1 lake env lean docs/SU2-TWO-TRANSPORTER-NOGO-ORACLE.lean
run_keep pr39-oracle-2 lake env lean docs/SU2-TWO-TRANSPORTER-NOGO-ORACLE.lean
run_keep pr39-core-build lake build YangMillsCore
git status --short
sha256sum lean-toolchain lakefile.lean lake-manifest.json docs/SU2-TWO-TRANSPORTER-NOGO-ORACLE.lean

cd "$RUN"
git clone --no-tags "$REPO" "$RUN/pr43"
cd "$RUN/pr43"
git fetch origin refs/pull/43/head
git checkout --detach "$PR43_AUDITED"
echo "PR43_AUDITED_HEAD=$(git rev-parse HEAD)"
echo "PR43_LAKE_PREEXISTS=$(test -e .lake && echo yes || echo no)"
cp lake-manifest.json "$RUN/pr43-manifest-before.json"
sha256sum lean-toolchain lakefile.lean lake-manifest.json
run_keep pr43-lake-update lake update
cmp -s lake-manifest.json "$RUN/pr43-manifest-before.json"
if [ "$?" -ne 0 ]; then echo "PR43_MANIFEST_UNCHANGED=no"; FAILURES=$((FAILURES+1)); else echo "PR43_MANIFEST_UNCHANGED=yes"; fi
run_keep pr43-cache-get lake exe cache get
run_keep pr43-import-1 lake env lean "$RUN/ImportMathlib.lean"
run_keep pr43-import-2 lake env lean "$RUN/ImportMathlib.lean"
run_keep pr43-oracle-build lake build YangMills.SU2ThetaPrism.Oracle
run_keep pr43-oracle lake env lean YangMills/SU2ThetaPrism/Oracle.lean
git status --short
sha256sum lean-toolchain lakefile.lean lake-manifest.json YangMills/SU2ThetaPrism/Oracle.lean

cd "$RUN"
git clone --no-tags "$REPO" "$RUN/pr39-repeat"
cd "$RUN/pr39-repeat"
git fetch origin refs/pull/39/head
git checkout --detach "$PR39"
echo "PR39_REPEAT_HEAD=$(git rev-parse HEAD)"
echo "PR39_REPEAT_LAKE_PREEXISTS=$(test -e .lake && echo yes || echo no)"
cp lake-manifest.json "$RUN/pr39-repeat-manifest-before.json"
run_keep pr39-repeat-lake-update lake update
cmp -s lake-manifest.json "$RUN/pr39-repeat-manifest-before.json"
if [ "$?" -ne 0 ]; then echo "PR39_REPEAT_MANIFEST_UNCHANGED=no"; FAILURES=$((FAILURES+1)); else echo "PR39_REPEAT_MANIFEST_UNCHANGED=yes"; fi
run_keep pr39-repeat-cache-get lake exe cache get
run_keep pr39-repeat-import-1 lake env lean "$RUN/ImportMathlib.lean"
run_keep pr39-repeat-import-2 lake env lean "$RUN/ImportMathlib.lean"
run_keep pr39-repeat-minimal-build lake build YangMills.OS.TwoTransporterHaarProjection
run_keep pr39-repeat-oracle-1 lake env lean docs/SU2-TWO-TRANSPORTER-NOGO-ORACLE.lean
run_keep pr39-repeat-oracle-2 lake env lean docs/SU2-TWO-TRANSPORTER-NOGO-ORACLE.lean
git status --short
sha256sum lean-toolchain lakefile.lean lake-manifest.json docs/SU2-TWO-TRANSPORTER-NOGO-ORACLE.lean

cd "$RUN"
echo "UTC_END=$(date -u +%Y-%m-%dT%H:%M:%SZ)"
echo "FAILURE_COUNT=$FAILURES"
sha256sum "$TRANSCRIPT"
exit "$FAILURES"
